# 📊 Power BI & Power Query Data Plan Architectural Assessment
### *Essential Pre-Implementation Discovery Checklist based on Fundamentals of Data Engineering (Reis & Housley)*

> **Target Platform:** Power BI Pro / Premium (Power Query M, Power BI Dataflows, VertiPaq Semantic Models, DAX)  
> **Methodology:** The Data Engineering Lifecycle (*Fundamentals of Data Engineering*, Chapter 2)  
> **Use Case Focus:** Web APIs, Open Data Portals (data.gov, Socrata, CKAN, US Census), REST endpoints, and Flat File ingestions  
> **Purpose:** Evaluate architectural feasibility, capacity limits, schema drift resilience, and operational risks **when building a pure Power BI enterprise data solution without external cloud platforms.**

---

## 🧭 The Power BI-Centric Data Engineering Lifecycle
When an organization decides to use **Power BI for everything**, the traditional data engineering lifecycle maps directly into Power BI's built-in ecosystem:

```
[ 1. Generation ] ───▶ [ 2. Storage ] ───▶ [ 3. Ingestion ] ───▶ [ 4. Transformation ] ───▶ [ 5. Serving (BI/Analytics) ]
  (Open Data API)        (Dataflows / ADLS)     (Power Query M Web)     (Power Query / M Engine)       (VertiPaq Model / DAX)
──────────────────────────────────────────────────────────────────────────────────────────────────────────
                       CROSS-CUTTING UNDERCURRENTS:
  • Security & Privacy (RLS)   • Data Governance (Certified Datasets)   • DataOps & ALM   • Refresh Quotas (Pro/PPU)
```

### ⚠️ The Golden Architectural Rule for "Power BI Only"
**Never build monolithic `.pbix` files that combine API extraction, transformation, data modeling, and report visuals into a single desktop file.**  
Always decouple the architecture into a 3-tier structure:
1. **Tier 1: Power BI Dataflow (Extraction & Bronze Staging):** Hits the Web API once, paginates, and archives raw snapshots into Power BI managed storage.
2. **Tier 2: Power BI Dataflow or Central Semantic Model (Silver/Gold):** Applies business transformations, handles schema drift, and builds the Star Schema (Dimensions & Facts).
3. **Tier 3: Thin Reports (Visualization):** Connect via **Live Connection** to the Certified Semantic Model. Multiple reports share one dataset with zero redundant API calls.


---
## 1. Generation / Source Systems (Book Reference: Page 36)
> *"Before even ingesting data, a data engineer must evaluate how the source systems generate data by answering these big questions about source systems."* — Reis & Housley

### 📋 Architectural Questions & Power BI / Power Query Context

1. **Schema Evolution Policy:** *If schema changes (say, a new column is added or removed), how is this dealt with and communicated to downstream stakeholders?*
   * **Power BI Reality:** Power Query is schema-rigid by default. Hardcoded steps like `Table.SelectColumns(Source, {"colA", "colB"})` or `Table.TransformColumnTypes` will fail if the API drops or renames a field.
   * **Power BI Best Practice:** Use defensive M patterns. Apply `MissingField.Ignore` in `Table.SelectColumns(Source, {"colA", "colB"}, MissingField.Ignore)`. Avoid hardcoded column type casting until after non-essential columns are pruned. Configure Power BI Service dataset refresh failure notifications to an administrative distribution list.
2. **Pull Frequency:** *How frequently should data be pulled from the source system?*
   * **Power BI Reality:** Power BI Pro licenses support a maximum of **8 scheduled refreshes per day**. Premium / PPU licenses support up to **48 per day**.
   * **Power BI Best Practice:** Open data portals (e.g., Socrata, data.gov, Census) usually publish data on daily, weekly, or monthly schedules. Align your refresh schedule to the portal's published cadence. Over-refreshing burns gateway/tenant concurrency without retrieving new data.
3. **Stateful Systems & CDC:** *For stateful systems (e.g., a database tracking customer account information), is data provided as periodic snapshots or update events from change data capture (CDC)? What’s the logic for how changes are performed, and how are these tracked in the source database?*
   * **Power BI Reality:** Open data portals **do not provide CDC streams**. They provide either full file downloads (CSV/JSON) or REST endpoints with query parameters (e.g., SODA API `$where=updated_at > '...'`).
   * **Power BI Best Practice:** If the dataset is large (millions of rows) with historical updates, configure **Power BI Incremental Refresh** using parameters `RangeStart` and `RangeEnd`. Power BI will partition historical data and only refresh the recent active window.
4. **Data Provider Transmission:** *Who/what is the data provider that will transmit the data for downstream consumption?*
   * **Power BI Reality:** Power BI operates as a **Pull** client via HTTP GET requests executed by the Mashup Engine.
   * **Power BI Best Practice:** **Mandatory M Pattern:** Never build dynamic URLs with string concatenation in `Web.Contents("https://api.../" & id)` because the Power BI Service cloud refresh engine will flag it as an unrefreshable dynamic data source. Always separate the base URL from dynamic paths and queries:
     `Web.Contents("https://api.data.gov", [RelativePath="resource/endpoint.json", Query=[#"$limit"="50000"]])`.
5. **Source System Impact & Rate Limiting:** *Will reading from a data source impact its performance?*
   * **Power BI Reality:** Public open data portals enforce strict rate limits (e.g., HTTP 429 Too Many Requests or 1,000 requests/hour for anonymous IP addresses).
   * **Power BI Best Practice:** Register for an official **App Token / API Key** (e.g., Socrata App Token, api.data.gov key) and pass it in the M header: `[Headers=[#"X-App-Token"="YOUR_TOKEN"]]`. Fetch in large batch page sizes (e.g., 50,000 rows) to minimize HTTP handshake overhead.
6. **Upstream Dependencies:** *Does the source system have upstream data dependencies? What are the characteristics of these upstream systems?*
   * **Power BI Reality:** US government agency feeds often experience release delays or publish dates days after data collection.
   * **Power BI Best Practice:** Query the portal's dataset metadata endpoint (e.g., `rowsUpdatedAt` or `data_as_of`) alongside the payload. Expose a DAX measure on the report canvas: `"Data Current As Of: " & SELECTEDVALUE(Metadata[LastUpdated])`.
7. **Data Quality Checks (Late / Missing Data):** *Are data-quality checks in place to check for late or missing data?*
   * **Power BI Reality:** Power BI does not have native data unit-testing frameworks like dbt or Great Expectations.
   * **Power BI Best Practice:** Build a "Circuit Breaker" in Power Query: if `Table.RowCount(Source) == 0`, throw an explicit M exception `error "Empty payload received from Open Data API - halting refresh to preserve existing data"`. This halts the refresh and leaves the previous valid dataset intact in the Power BI Service.


In [ ]:
# ==============================================================================
# 🛠️ STAGE 1: SOURCE SYSTEM DISCOVERY MATRIX (POWER BI FOCUS)
# ==============================================================================
import pandas as pd

pbi_source_assessment = [
    {
        "ID": "1.1",
        "Category": "Generation",
        "Question": "Schema Evolution Policy & Communication",
        "Status": "Action Required", # Verified | In Progress | Action Required | Not Applicable
        "Answer_Details": "Open data portal frequently adds new metadata fields and occasionally renames location headers.",
        "Risk_Level": "HIGH",        # LOW | MEDIUM | HIGH | CRITICAL
        "PBI_Mitigation": "Use Table.SelectColumns with MissingField.Ignore in Power Query M to prevent refresh crashes."
    },
    {
        "ID": "1.2",
        "Category": "Generation",
        "Question": "Pull Frequency & Refresh Cadence",
        "Status": "Verified",
        "Answer_Details": "Source dataset updates nightly at 03:00 EST; scheduled refresh configured for 05:00 EST (1 of 8 daily slots).",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Configured single daily scheduled refresh in Power BI Service workspace."
    },
    {
        "ID": "1.3",
        "Category": "Generation",
        "Question": "Stateful Systems vs Incremental CDC",
        "Status": "In Progress",
        "Answer_Details": "Dataset contains 3 million rows with a 'last_modified_date' field; full refresh takes 45 minutes.",
        "Risk_Level": "MEDIUM",
        "PBI_Mitigation": "Implement Power BI Incremental Refresh using RangeStart and RangeEnd M parameters."
    },
    {
        "ID": "1.4",
        "Category": "Generation",
        "Question": "Data Provider Transmission & Web.Contents Syntax",
        "Status": "Action Required",
        "Answer_Details": "Original developer used dynamic URL string concatenation; scheduled refresh fails in Service with 'Dynamic data sources' error.",
        "Risk_Level": "CRITICAL",
        "PBI_Mitigation": "Refactor M query to use Web.Contents with RelativePath and Query options."
    },
    {
        "ID": "1.5",
        "Category": "Generation",
        "Question": "Source System Impact & Rate Limiting (HTTP 429)",
        "Status": "Verified",
        "Answer_Details": "Acquired registered Socrata App Token; quota increased to 50,000 requests/day.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Passed App Token via headers in Web.Contents."
    },
    {
        "ID": "1.6",
        "Category": "Generation",
        "Question": "Upstream Dependency Characteristics",
        "Status": "In Progress",
        "Answer_Details": "Agency occasionally misses weekend updates; Monday report may show Friday data.",
        "Risk_Level": "MEDIUM",
        "PBI_Mitigation": "Extract portal metadata timestamp and display 'Data Current As Of' card on executive report."
    },
    {
        "ID": "1.7",
        "Category": "Generation",
        "Question": "Late / Missing Data Quality Circuit Breaker",
        "Status": "Action Required",
        "Answer_Details": "API outage once caused an empty table to overwrite the existing semantic model.",
        "Risk_Level": "HIGH",
        "PBI_Mitigation": "Add Power Query circuit breaker step asserting Table.RowCount > 0 before returning data."
    }
]

df_pbi_source = pd.DataFrame(pbi_source_assessment)
display(df_pbi_source)


---
## 2. Storage Systems (Book Reference: Page 38)
> *"Evaluating storage requires looking at write/read speeds, bottlenecks, and future scale."* — Reis & Housley

### 📋 Architectural Questions & Power BI / VertiPaq Context

1. **Read/Write Speed Compatibility:** *Is this storage solution compatible with the architecture’s required write and read speeds?*
   * **Power BI Context:** Power BI uses **VertiPaq**, a columnar, in-memory, compressed analytical database. Read speeds are blazing fast for aggregation queries. However, write speed is batch-oriented (during data refresh). VertiPaq is **not** an operational read/write store.
2. **Downstream Bottlenecks:** *Will storage create a bottleneck for downstream processes?*
   * **Power BI Context:** In VertiPaq, bottlenecks are caused by **high-cardinality columns** (e.g., unique GUIDs, unrounded timestamps, high-precision decimals) and bidirectional table relationships. These bloat dictionary encoding and slow down DAX calculations.
3. **Technology Mechanics & "Unnatural Acts":** *Do you understand how this storage technology works? Are you utilizing the storage system optimally or committing "unnatural acts" (like applying a high rate of random access updates in an object storage system)?*
   * **Power BI Context:** Committing "unnatural acts" in Power BI includes:
     - Building single, massive flat tables with 100+ columns instead of a clean **Star Schema** (Facts & Dimensions).
     - Importing row-level timestamps (seconds/milliseconds) that destroy column compression.
     - Using Power BI as a data export tool to dump millions of raw rows into Excel.
4. **Anticipated Future Scale:** *Will this storage system handle anticipated future scale? (Consider total available storage, read operation rate, write volume, etc.)*
   * **Power BI Context:** 
     - **Power BI Pro:** Maximum compressed dataset size is **1 GB**. Maximum workspace storage is 10 GB.
     - **Power BI PPU / Premium Capacity:** Supports Large Dataset Storage format (up to 100 GB - 400 GB).
     - **Memory Rule:** During data refresh, Power BI requires **2x the dataset size in RAM** to build and swap the new VertiPaq model.
5. **Downstream SLA & Report Response Time:** *Will downstream users and processes be able to retrieve data within the required service-level agreement (SLA)?*
   * **Power BI Context:** Standard visual interactions must render under 2-3 seconds. Optimize DAX measures using DAX Studio; avoid slow row-by-row iterators (`SUMX`, `FILTER` over entire fact tables) when simple vectorized aggregations suffice.
6. **Metadata & Lineage:** *Are you capturing metadata about schema evolution, data flows, and data lineage?*
   * **Power BI Context:** Use the **Power BI Service Lineage View** to map dependencies from Source API $	o$ Dataflow $	o$ Semantic Model $	o$ Reports $	o$ Dashboards.
7. **Pure Storage vs Complex Queries:** *Is this a pure storage solution (object storage) or does it support complex query patterns (like a cloud data warehouse)?*
   * **Power BI Context:** VertiPaq is built specifically for OLAP multidimensional analytical queries (DAX). It does not support arbitrary write operations or complex transactional stored procedures.
8. **Schema Rigidity:** *Is the storage system schema-agnostic, flexible schema, or enforced schema?*
   * **Power BI Context:** VertiPaq requires strict column schemas and strongly typed data types. Every column must be explicitly cast (text, integer, decimal, boolean).
9. **Master Data & Golden Records:** *How are you tracking master data, golden records, data quality, and data lineage for data governance?*
   * **Power BI Context:** Publish and **Certify** the core Semantic Model in the Power BI Service. Disable report creators from creating their own local data models to ensure a **Single Version of the Truth**.
10. **Regulatory Compliance & Data Sovereignty:** *How are you handling regulatory compliance and data sovereignty (e.g., storing data in specific geographical locations)?*
    * **Power BI Context:** Ensure the Power BI Tenant or workspace is hosted in the approved Azure region (e.g., US Gov Cloud, East US). Verify that open data does not contain accidentally exposed PII (Personally Identifiable Information).


In [ ]:
# ==============================================================================
# 🛠️ STAGE 2: STORAGE & VERTIPAQ DISCOVERY MATRIX
# ==============================================================================
pbi_storage_assessment = [
    {
        "ID": "2.1",
        "Category": "Storage",
        "Question": "Read/Write Speed Compatibility (VertiPaq)",
        "Status": "Verified",
        "Answer_Details": "VertiPaq columnar memory format provides sub-second aggregations for 5 million rows.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Model data as Import mode Star Schema."
    },
    {
        "ID": "2.2",
        "Category": "Storage",
        "Question": "Downstream Bottlenecks & High-Cardinality Columns",
        "Status": "Action Required",
        "Answer_Details": "Dataset contains 3 unique transaction hash columns consuming 65% of memory.",
        "Risk_Level": "HIGH",
        "PBI_Mitigation": "Remove unnecessary transaction hashes in Power Query; retain only surrogate keys."
    },
    {
        "ID": "2.3",
        "Category": "Storage",
        "Question": "Storage Mechanics & Star Schema Modeling",
        "Status": "In Progress",
        "Answer_Details": "Initial model is a single 80-column wide flat table causing DAX calculation lag.",
        "Risk_Level": "HIGH",
        "PBI_Mitigation": "Deconstruct flat table into 1 Fact Table (Sales) and 4 Dimension Tables (Date, Agency, Program, Geography)."
    },
    {
        "ID": "2.4",
        "Category": "Storage",
        "Question": "Anticipated Scale & Pro 1 GB Limit",
        "Status": "Verified",
        "Answer_Details": "Compressed model is 320 MB; comfortably below the 1 GB Pro limit for the next 2 years.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Run VertiPaq Analyzer in DAX Studio quarterly to monitor memory growth."
    },
    {
        "ID": "2.5",
        "Category": "Storage",
        "Question": "Downstream SLA & Visual Rendering Speed",
        "Status": "Verified",
        "Answer_Details": "All visual interactions complete under 1.5 seconds in testing.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Validate DAX measures using Performance Analyzer in Power BI Desktop."
    },
    {
        "ID": "2.6",
        "Category": "Storage",
        "Question": "Metadata & Lineage Tracking",
        "Status": "Verified",
        "Answer_Details": "Dependencies tracked via Power BI Workspace Lineage View.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Document workspace relationships in central wiki."
    },
    {
        "ID": "2.7",
        "Category": "Storage",
        "Question": "Pure Storage vs Analytical Cache (VertiPaq)",
        "Status": "Verified",
        "Answer_Details": "Dataflow used for storage cache; Semantic Model used for analytical serving.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Maintain clear separation between Dataflow and Semantic Model."
    },
    {
        "ID": "2.8",
        "Category": "Storage",
        "Question": "Schema Rigidity & Type Enforcement",
        "Status": "Verified",
        "Answer_Details": "All columns explicitly cast in Power Query before loading into VertiPaq.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Ensure no 'Any' types remain in the final Power Query output."
    },
    {
        "ID": "2.9",
        "Category": "Storage",
        "Question": "Master Data & Golden Dataset Certification",
        "Status": "In Progress",
        "Answer_Details": "Semantic Model needs to be endorsed as 'Certified' in Power BI Service.",
        "Risk_Level": "MEDIUM",
        "PBI_Mitigation": "Apply workspace admin certification to prevent duplicate model sprawl."
    },
    {
        "ID": "2.10",
        "Category": "Storage",
        "Question": "Data Sovereignty & Privacy Compliance",
        "Status": "Verified",
        "Answer_Details": "Public open data contains no PII; tenant hosted in US East region.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Verify data policy compliance before each major release."
    }
]

df_pbi_storage = pd.DataFrame(pbi_storage_assessment)
display(df_pbi_storage)


---
## 3. Ingestion Phase (Book Reference: Page 40)
> *"The ingestion stage is often a major bottleneck, making these considerations crucial."* — Reis & Housley

### 📋 Architectural Questions & Power BI / Power Query Context

1. **Use Cases & Reusability:** *What are the use cases for the data I’m ingesting? Can I reuse this data rather than create multiple versions of the same dataset?*
   * **Power BI Reality:** If multiple departments need this open data, duplicating the API extraction in multiple `.pbix` files will cause rate-limit bans and conflicting metrics.
   * **Power BI Best Practice:** Create a **Power BI Dataflow (Gen1)** in a shared workspace. Ingest and stage the data once. All reports and semantic models across the company then connect to this single Dataflow entity using the Power BI Dataflow connector.
2. **Generation & Ingestion Reliability:** *Are the systems generating and ingesting this data reliably, and is the data available when I need it?*
   * **Power BI Reality:** Power Query web requests do not have native exponential backoff or automatic retry loops built-in. If an API call suffers a momentary network blip, the scheduled refresh fails.
   * **Power BI Best Practice:** In Power Query M, wrap transient web calls in `try ... otherwise` blocks, or split extraction across smaller daily chunks to avoid all-or-nothing failures.
3. **Data Destination After Ingestion:** *What is the data destination after ingestion?*
   * **Power BI Reality:** Raw API data should land in a **Power BI Dataflow** (stored as CDM/Parquet in Power BI managed Azure storage), not directly in report visual caches.
4. **Access Frequency Profile:** *How frequently will I need to access the data?*
   * **Power BI Reality:** The API is accessed only during scheduled refresh windows (e.g., once every 24 hours). End-users interact with the in-memory VertiPaq model, not the live API.
5. **Arrival Volume & API Pagination:** *In what volume will the data typically arrive?*
   * **Power BI Reality:** **The 2-Hour Ceiling:** Power BI Pro has a **2-hour refresh limit** per dataset (5 hours for Premium). Open data APIs often return a maximum of 1,000 to 50,000 records per page. Paginating through millions of records via recursive M functions (`List.Generate`) can exhaust memory and trigger a timeout.
   * **Power BI Best Practice:** 
     - If the portal provides a direct bulk export (e.g. daily CSV or GZ file), ingest the bulk file instead of paginating through thousands of REST JSON calls.
     - If REST is required, limit extraction to rolling 90 days and use Incremental Refresh.
6. **Data Format Compatibility:** *What format is the data in? Can my downstream storage and transformation systems handle this format?*
   * **Power BI Reality:** Open data portals usually provide JSON, GeoJSON, CSV, or XML.
   * **Power BI Best Practice:** Power Query M natively parses JSON via `Json.Document` and CSV via `Csv.Document`. Unpack nested JSON arrays into flat tabular rows early in the pipeline.
7. **Source Data Usability:** *Is the source data in good shape for immediate downstream use? If so, for how long, and what may cause it to be unusable?*
   * **Power BI Reality:** Open data portals frequently suffer from inconsistent date formats (e.g., mix of ISO8601 and US dates), null values in mandatory fields, and text-encoded numbers (`"$1,200.50"`).
   * **Power BI Best Practice:** Standardize all parsing in Power Query: trim strings, remove currency symbols, parse dates with explicit culture (e.g. `"en-US"`), and replace empty strings with `null`.
8. **In-Flight Transformations:** *If the data is from a streaming source, does it need to be transformed before reaching its destination (e.g., in-flight transformations)?*
   * **Power BI Reality:** Power Query is a batch engine. For real-time open data (e.g., live transit feeds), standard Power BI import will not suffice; you must use Power BI Streaming/Push Datasets.


In [ ]:
# ==============================================================================
# 🛠️ STAGE 3: INGESTION PHASE DISCOVERY MATRIX (POWER BI FOCUS)
# ==============================================================================
pbi_ingestion_assessment = [
    {
        "ID": "3.1",
        "Category": "Ingestion",
        "Question": "Use Case Clarity & Dataflow Reusability",
        "Status": "Verified",
        "Answer_Details": "Single Power BI Dataflow created in shared workspace; 3 departmental models consume this single entity.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Enforce policy that all reports connect to Dataflow, not directly to Open Data API."
    },
    {
        "ID": "3.2",
        "Category": "Ingestion",
        "Question": "Ingestion Reliability & Transient Failures",
        "Status": "Action Required",
        "Answer_Details": "Occasional HTTP 504 Gateway Timeout from public API during scheduled refresh.",
        "Risk_Level": "HIGH",
        "PBI_Mitigation": "Build custom M try/otherwise wrapper with 30-second delay or switch to bulk CSV endpoint."
    },
    {
        "ID": "3.3",
        "Category": "Ingestion",
        "Question": "Clear Ingestion Landing Destination",
        "Status": "Verified",
        "Answer_Details": "Data lands in Power BI Dataflow staging entity before ingestion into Semantic Model.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Maintain dedicated Staging workspace."
    },
    {
        "ID": "3.4",
        "Category": "Ingestion",
        "Question": "Access Frequency vs API Load",
        "Status": "Verified",
        "Answer_Details": "API polled once per day at 04:00 EST; users query cached VertiPaq model.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Schedule single daily refresh slot."
    },
    {
        "ID": "3.5",
        "Category": "Ingestion",
        "Question": "Arrival Volume & 2-Hour Refresh Limit",
        "Status": "Action Required",
        "Answer_Details": "Paginating 4 million records in Power Query takes 1 hour 45 minutes, approaching 2-hour Pro cap.",
        "Risk_Level": "CRITICAL",
        "PBI_Mitigation": "Switch from paginated JSON endpoint to portal's daily bulk CSV download endpoint (drops duration to 8 mins)."
    },
    {
        "ID": "3.6",
        "Category": "Ingestion",
        "Question": "Format Compatibility & Unnesting",
        "Status": "Verified",
        "Answer_Details": "Nested JSON payloads expanded using Table.ExpandRecordColumn in Power Query.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Optimize column expansion step to select only required attributes."
    },
    {
        "ID": "3.7",
        "Category": "Ingestion",
        "Question": "Source Data Shelf-Life & Type Cleansing",
        "Status": "Verified",
        "Answer_Details": "Parsed currency strings, stripped commas, and converted dates using 'en-US' locale.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Standardize type-casting steps in Dataflow."
    },
    {
        "ID": "3.8",
        "Category": "Ingestion",
        "Question": "In-Flight Transformation Requirements",
        "Status": "Not Applicable",
        "Answer_Details": "Batch daily reporting only; no sub-minute streaming requirements.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "N/A - Standard batch scheduled refresh."
    }
]

df_pbi_ingestion = pd.DataFrame(pbi_ingestion_assessment)
display(df_pbi_ingestion)


---
## 4. Transformation Phase (Book Reference: Page 43)
> *"Transformations are where data begins to create tangible value, which requires asking these fundamental questions."* — Reis & Housley

### 📋 Architectural Questions & Power BI / Power Query Context

1. **Cost, ROI & Business Value:** *What’s the cost and return on investment (ROI) of the transformation? What is the associated business value?*
   * **Power BI Reality:** Because the team is using existing Power BI Pro licenses ($10/user/month), software licensing cost is low. However, **engineering maintenance cost** can skyrocket if transformations are fragile and constantly break.
   * **Power BI Best Practice:** Keep business value clear: does this transformation automate a manual 4-hour Excel task? If yes, the ROI is massive. Avoid building complex statistical transformations in Power Query that could easily be calculated dynamically in DAX.
2. **Simplicity, Isolation & Query Folding:** *Is the transformation as simple and self-isolated as possible?*
   * **Power BI Reality:** **Web APIs do not support Query Folding.** Every filter, join, and aggregation in Power Query will be executed by your local machine or the Power BI Service cloud Mashup Engine in memory.
   * **Power BI Best Practice:** 
     - Push filters into the API query parameters whenever possible (e.g., Socrata `$where` or `$select` parameters) so the API server does the heavy filtering before transmitting data.
     - Keep M scripts modular: separate steps into logical phases: Extraction $	o$ Cleaning $	o$ Shaping $	o$ Typing.
3. **Business Rules Support (M vs DAX):** *What business rules do the transformations support?*
   * **Power BI Reality:** Blurring the line between Power Query (M) and DAX creates maintenance nightmares.
   * **The Clean Boundary:**
     - **Power Query (M):** Responsible for structural transformations, data cleaning, filtering historical ranges, and shaping tables into Star Schema Dimensions and Facts.
     - **DAX:** Responsible for dynamic business logic, KPIs, ratios, time intelligence (Year-to-Date, YoY growth), and user filter context.


In [ ]:
# ==============================================================================
# 🛠️ STAGE 4: TRANSFORMATION PHASE DISCOVERY MATRIX (POWER BI FOCUS)
# ==============================================================================
pbi_transformation_assessment = [
    {
        "ID": "4.1",
        "Category": "Transformation",
        "Question": "Cost, ROI & Associated Business Value",
        "Status": "Verified",
        "Answer_Details": "Automates weekly manual agency reporting, saving 15 analyst hours/week with zero new license costs.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Track report usage in Power BI Service Usage Metrics."
    },
    {
        "ID": "4.2",
        "Category": "Transformation",
        "Question": "Simplicity, Isolation & Query Folding Limitations",
        "Status": "Action Required",
        "Answer_Details": "Power Query performs in-memory joins between two 1-million row API queries, consuming excessive memory.",
        "Risk_Level": "HIGH",
        "PBI_Mitigation": "Push filtering to source API query parameters ($select, $where) and avoid large table merges in Power Query."
    },
    {
        "ID": "4.3",
        "Category": "Transformation",
        "Question": "Clear Separation of M vs DAX",
        "Status": "Verified",
        "Answer_Details": "Power Query shapes Star Schema; all KPIs (YoY growth, budget variance) written in DAX.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Document DAX measure dictionary in central data catalog."
    }
]

df_pbi_transformation = pd.DataFrame(pbi_transformation_assessment)
display(df_pbi_transformation)


---
## 5. Serving Data (Specific to Analytics & Machine Learning) (Book Reference: Page 46)
> *"While serving data covers analytics, ML, and reverse ETL, the authors highlight these considerations specifically for analytics and ML deployments."* — Reis & Housley

### 📋 Architectural Questions & Power BI Context

1. **Feature Engineering & Analytics Data Quality:** *Is the data of sufficient quality to perform reliable analysis or feature engineering?*
   * **Power BI Context:** Ensure referential integrity between Fact tables and Dimension tables. If a foreign key in Fact Sales has no match in Dim Customer, Power BI will inject an empty blank row into the dimension, confusing business users.
2. **Discoverability:** *Is the data discoverable? Can analysts easily find valuable certified datasets?*
   * **Power BI Context:** Enable the **Power BI Service Data Hub**. Apply the official **"Certified"** endorsement badge to the enterprise semantic model so self-service analysts build new reports on trusted data.
3. **Technical & Organizational Boundaries:** *Where are the technical and organizational boundaries between data engineering and reporting?*
   * **Power BI Context:** Enforce the **Dataset vs Report Separation Principle**:
     - Data Engineer owns: Source API connectors, Dataflows, Data Model relationships, security (RLS), and certified DAX base measures.
     - Business Analyst / Report Author owns: Visual layouts, report pages, bookmarks, and user presentation.
     - Authors connect via **Power BI Live Connection** (Thin Reports).
4. **Ground Truth & Bias Representation:** *Does the dataset properly represent ground truth? Is it unfairly biased?*
   * **Power BI Context:** Open data often has historical reporting gaps (e.g., changes in reporting standards between presidential administrations, under-reporting by certain municipalities). Document known portal caveats on an **Info / Assumptions page** in the Power BI report.


In [ ]:
# ==============================================================================
# 🛠️ STAGE 5: SERVING DATA DISCOVERY MATRIX (POWER BI FOCUS)
# ==============================================================================
pbi_serving_assessment = [
    {
        "ID": "5.1",
        "Category": "Serving",
        "Question": "Data Quality & Dimensional Integrity",
        "Status": "In Progress",
        "Answer_Details": "Fact table contains 2% orphan records with agency codes not found in Agency Dimension.",
        "Risk_Level": "MEDIUM",
        "PBI_Mitigation": "Add an 'Unknown Agency' row in Dimension table in Power Query to catch orphan keys."
    },
    {
        "ID": "5.2",
        "Category": "Serving",
        "Question": "Discoverability & Data Hub Certification",
        "Status": "Verified",
        "Answer_Details": "Model endorsed as 'Certified' in the Enterprise Workspace Data Hub.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Enable workspace discovery settings in Power BI Admin Portal."
    },
    {
        "ID": "5.3",
        "Category": "Serving",
        "Question": "DE vs Analyst Organizational Boundary",
        "Status": "Verified",
        "Answer_Details": "Clean boundary enforced: Data Engineers maintain shared dataset; analysts author thin reports via Live Connection.",
        "Risk_Level": "LOW",
        "PBI_Mitigation": "Grant 'Build' permission to analysts without granting direct workspace edit rights."
    },
    {
        "ID": "5.4",
        "Category": "Serving",
        "Question": "Ground Truth & Open Data Caveats",
        "Status": "Action Required",
        "Answer_Details": "2020 pandemic data reflects severe anomaly compared to historical baseline; stakeholders may misinterpret trends.",
        "Risk_Level": "MEDIUM",
        "PBI_Mitigation": "Add dedicated 'Methodology & Data Disclaimers' pop-up dialog on executive dashboard."
    }
]

df_pbi_serving = pd.DataFrame(pbi_serving_assessment)
display(df_pbi_serving)


---
## 6. Executive Readiness Scorecard & Risk Summary
This cell aggregates your assessment across all 5 lifecycle phases, calculates your **Power BI Data Architecture Readiness Score (0-100%)**, highlights blocking items, and produces an actionable mitigation summary.


In [ ]:
# ==============================================================================
# 📊 CONSOLIDATED POWER BI READINESS SCORECARD
# ==============================================================================
import pandas as pd
import numpy as np

all_pbi_assessments = (
    pbi_source_assessment + 
    pbi_storage_assessment + 
    pbi_ingestion_assessment + 
    pbi_transformation_assessment + 
    pbi_serving_assessment
)
df_pbi_full = pd.DataFrame(all_pbi_assessments)

status_map = {"Verified": 100, "In Progress": 50, "Action Required": 0, "Not Applicable": 100}
df_pbi_full["Status_Score"] = df_pbi_full["Status"].map(status_map)

total_q = len(df_pbi_full)
verified_q = (df_pbi_full["Status"] == "Verified").sum()
in_progress_q = (df_pbi_full["Status"] == "In Progress").sum()
action_required_q = (df_pbi_full["Status"] == "Action Required").sum()
critical_r = (df_pbi_full["Risk_Level"] == "CRITICAL").sum()
high_r = (df_pbi_full["Risk_Level"] == "HIGH").sum()

pbi_score = round(df_pbi_full["Status_Score"].mean(), 1)

print("=" * 80)
print("📊 POWER BI DATA PLAN ARCHITECTURAL READINESS REPORT")
print("=" * 80)
print(f"Overall Architectural Readiness Score: {pbi_score} / 100")
print(f"Total Questions Evaluated:            {total_q}")
print(f"  • Verified & Ready:                  {verified_q}")
print(f"  • In Progress:                       {in_progress_q}")
print(f"  • Action Required (Blocking):        {action_required_q}")
print(f"Risk Breakdown:")
print(f"  • Critical Risks:                    {critical_r}")
print(f"  • High Risks:                        {high_r}")
print("=" * 80)

if pbi_score >= 85 and critical_r == 0:
    print("🟢 STATUS: GREEN (Ready for Implementation)")
    print("   Architecture adheres to Power BI best practices. Proceed with build.")
elif pbi_score >= 65 and critical_r == 0:
    print("🟡 STATUS: YELLOW (Conditional Approval)")
    print("   Resolve 'Action Required' items before publishing dataset to production.")
else:
    print("🔴 STATUS: RED (Implementation Blocked)")
    print("   Critical architectural traps detected (e.g., dynamic URLs, timeout risks).")
    print("   Fix Critical/High risks before writing Power BI reports.")

print("=" * 80)

print("\n📈 READINESS SCORE BY LIFECYCLE STAGE:")
df_pbi_cat = df_pbi_full.groupby("Category").agg(
    Questions=("ID", "count"),
    Avg_Readiness=("Status_Score", "mean"),
    Action_Required=("Status", lambda s: (s == "Action Required").sum()),
    High_Or_Critical_Risks=("Risk_Level", lambda r: r.isin(["HIGH", "CRITICAL"]).sum())
).round(1).reset_index()

display(df_pbi_cat)

print("\n🚨 BLOCKING & HIGH-PRIORITY ACTION ITEMS:")
df_pbi_urgent = df_pbi_full[df_pbi_full["Risk_Level"].isin(["CRITICAL", "HIGH"]) | (df_pbi_full["Status"] == "Action Required")][
    ["ID", "Category", "Question", "Status", "Risk_Level", "PBI_Mitigation"]
]
display(df_pbi_urgent)


---
## 7. 🛠️ Power Query M Code Blueprints for Open Data APIs
These ready-to-use M templates address the most common technical traps identified in this assessment.


In [ ]:
# ==============================================================================
# 💡 BLUEPRINT 1: REFRESHABLE WEB.CONTENTS PATTERN (NO DYNAMIC URL ERROR)
# ==============================================================================
m_code_safe_web = '''
let
    // Base URL must be static for Power BI Service anonymous authentication
    BaseUrl = "https://data.cityofnewyork.us",
    Endpoint = "resource/erm2-nwe9.json",
    
    // Pass query parameters and headers safely
    Source = Json.Document(
        Web.Contents(
            BaseUrl,
            [
                RelativePath = Endpoint,
                Query = [
                    #"$limit" = "50000",
                    #"$where" = "created_date >= '2026-01-01T00:00:00.000'"
                ],
                Headers = [
                    #"X-App-Token" = "YOUR_APP_TOKEN_HERE"
                ]
            ]
        )
    ),
    
    // Circuit Breaker: Throw error if empty to avoid wiping out previous data
    Validate = if List.IsEmpty(Source) then error "API returned 0 rows - halting refresh" else Source,
    
    // Convert to Table and Expand
    ToTable = Table.FromList(Validate, Splitter.SplitByNothing(), null, null, ExtraValues.Error),
    Expanded = Table.ExpandRecordColumn(ToTable, "Column1", {"unique_key", "created_date", "agency", "complaint_type"}, MissingField.Ignore)
in
    Expanded
'''
print("Power Query M Blueprint 1 (Safe Web.Contents) loaded.")


In [ ]:
# ==============================================================================
# 💡 BLUEPRINT 2: DEFENSIVE SCHEMA SELECTION (RESILIENT TO API CHANGES)
# ==============================================================================
m_code_defensive_schema = '''
let
    Source = RawApiPayload,
    
    // Columns we desire
    DesiredColumns = {"id", "title", "department", "amount", "record_date"},
    
    // Safe Selection: Won't crash if an API field is omitted or renamed!
    SafeSelect = Table.SelectColumns(Source, DesiredColumns, MissingField.Ignore),
    
    // Safe Type Transformation: Only types columns that actually exist
    ExistingColumns = Table.ColumnNames(SafeSelect),
    TypeTransformations = List.Select(
        {
            {"id", type text},
            {"title", type text},
            {"department", type text},
            {"amount", Currency.Type},
            {"record_date", type datetime}
        },
        each List.Contains(ExistingColumns, _{0})
    ),
    TypedTable = Table.TransformColumnTypes(SafeSelect, TypeTransformations)
in
    TypedTable
'''
print("Power Query M Blueprint 2 (Defensive Schema) loaded.")
